# Final Individual Project: One-Way ANOVA Analysis

**Dataset:** `cleaned_yrbs_master.csv` | **Target Audience:** US High School Students  
**Responsible Person:** 李宥宣 (`113370237`)  
---

### Objective
The primary goal of this notebook is to perform a **One-Way ANOVA** using the processed master dataset. We focus on:
1. **Loading** the clean master dataset (`cleaned_yrbs_master.csv`).
2. **Validating** the sample sizes and descriptive statistics for each threat group.
3. **Conducting** the One-Way ANOVA test using `scipy.stats`.
4. **Visualizing** the results (Boxplot, Violin Plot, Mean Plot with 95% CI).
5. **Exporting** an automated statistical summary report and tables to the designated folders.

### Research Question & Hypotheses

**Research Question:** Is there a significant difference in BMI percentiles among students based on the frequency they were threatened or injured with a weapon on school property?

* **Null Hypothesis (H0):** The mean BMI percentiles are equal across all weapon threat frequency groups ($\mu_0 = \mu_1 = \mu_{2+}$).
* **Alternative Hypothesis (H1):** At least one group has a significantly different mean BMI percentile compared to the others.
* **Significance Level (Alpha):** 0.05

In [31]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# ==========================================================
# 1. Environment Setup and Loading
# ==========================================================
# Define output paths relative to the notebooks directory
input_data_path = '../data/processed/cleaned_yrbs_master.csv'
figures_dir = '../outputs/figures/'
tables_dir = '../outputs/tables/'
summary_dir = '../outputs/summary/'

# Ensure all required directories exist (create if they don't)
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)
os.makedirs(summary_dir, exist_ok=True)

# Load the Master CSV file
df = pd.read_csv(input_data_path)
print(f"✅ Master data loaded successfully. Total sample size for ANOVA: {len(df)}")

# Ensure the sorting logic of 'Threat_Group' is correct
group_order = ['0 times', '1 time', '2+ times']
df['Threat_Group'] = pd.Categorical(df['Threat_Group'], categories=group_order, ordered=True)

✅ Master data loaded successfully. Total sample size for ANOVA: 11541


In [33]:
# ==========================================================
# 2. Descriptive Statistics & Table Export
# ==========================================================
print("\n[Step 1] Descriptive Statistics of BMI Percentile by Group:")
print("=" * 60)

# Calculate sample size, mean, standard deviation, and median for each group
desc_stats = df.groupby('Threat_Group', observed=False)['BMIPCT'].agg(['count', 'mean', 'std', 'median']).reset_index()
desc_stats.columns = ['Threat_Group', 'Sample_Size', 'Mean_BMI_Pct', 'Std_Dev', 'Median']

# Print results to the console
for idx, row in desc_stats.iterrows():
    print(f"Group '{row['Threat_Group']}':")
    print(f"  - Sample Size (N): {int(row['Sample_Size'])}")
    print(f"  - Mean BMI Pct:    {row['Mean_BMI_Pct']:.2f}")
    print(f"  - Std Dev (SD):    {row['Std_Dev']:.2f}")
    print(f"  - Median:          {row['Median']:.2f}\n")
print("=" * 60)

# Export the descriptive statistics table to the tables folder
desc_table_path = os.path.join(tables_dir, '01_descriptive_statistics.csv')
desc_stats.to_csv(desc_table_path, index=False)
print(f"📁 Descriptive statistics table saved to: {desc_table_path}")


[Step 1] Descriptive Statistics of BMI Percentile by Group:
Group '0 times':
  - Sample Size (N): 10648
  - Mean BMI Pct:    64.83
  - Std Dev (SD):    27.38
  - Median:          70.12

Group '1 time':
  - Sample Size (N): 397
  - Mean BMI Pct:    67.55
  - Std Dev (SD):    27.81
  - Median:          75.12

Group '2+ times':
  - Sample Size (N): 496
  - Mean BMI Pct:    69.57
  - Std Dev (SD):    27.09
  - Median:          75.16

📁 Descriptive statistics table saved to: ../outputs/tables/01_descriptive_statistics.csv


In [35]:
# ==========================================================
# 3. One-Way ANOVA Analysis
# ==========================================================
# Split the BMIPCT data into three independent arrays based on Threat_Group
g0 = df[df['Threat_Group'] == '0 times']['BMIPCT'].values
g1 = df[df['Threat_Group'] == '1 time']['BMIPCT'].values
g2 = df[df['Threat_Group'] == '2+ times']['BMIPCT'].values

# Perform the One-Way ANOVA test
f_stat, p_val = stats.f_oneway(g0, g1, g2)

print("\n[Step 2] One-Way ANOVA Test Results:")
print("=" * 50)
print(f"F-Statistic : {f_stat:.4f}")
print(f"P-value     : {p_val:.6f}")
print("=" * 50)

# Statistical decision
alpha = 0.05
if p_val < alpha:
    print(f"🏆 Conclusion: Reject the null hypothesis (H0) since p < {alpha}.")
    print("There is a STATISTICALLY SIGNIFICANT difference in mean BMI percentiles among the threat groups.")
else:
    print(f"Conclusion: Fail to reject the null hypothesis (H0) since p >= {alpha}.")
    print("There is NO statistically significant difference in mean BMI percentiles among the threat groups.")


[Step 2] One-Way ANOVA Test Results:
F-Statistic : 8.6982
P-value     : 0.000168
🏆 Conclusion: Reject the null hypothesis (H0) since p < 0.05.
There is a STATISTICALLY SIGNIFICANT difference in mean BMI percentiles among the threat groups.


In [37]:
# ==========================================================
# 4. Statistical Visualization & Export
# ==========================================================
# Set plotting style
sns.set_theme(style="whitegrid")
group_colors = ["#4A90E2", "#50E3C2", "#E2847A"] 

print("\n[Step 3] Generating and Saving High-Resolution Visualizations...")

# (1) Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(x='Threat_Group', y='BMIPCT', data=df, palette=group_colors, hue='Threat_Group', legend=False)
plt.title('BMI Percentile Distribution by Weapon Threat Frequency', fontsize=14, pad=15)
plt.xlabel('Weapon Threat Frequency on School Property', fontsize=12, labelpad=10)
plt.ylabel('BMI Percentile (BMIPCT)', fontsize=12, labelpad=10)
plt.tight_layout()
boxplot_path = os.path.join(figures_dir, '01_bmi_threat_boxplot.png')
plt.savefig(boxplot_path, dpi=300)
plt.close()
print(f"  ✅ Boxplot saved to: {boxplot_path}")

# (2) Violin Plot
plt.figure(figsize=(8, 6))
sns.violinplot(x='Threat_Group', y='BMIPCT', data=df, palette=group_colors, hue='Threat_Group', legend=False, inner="quart")
plt.title('BMI Percentile Density by Weapon Threat Frequency', fontsize=14, pad=15)
plt.xlabel('Weapon Threat Frequency on School Property', fontsize=12, labelpad=10)
plt.ylabel('BMI Percentile (BMIPCT)', fontsize=12, labelpad=10)
plt.tight_layout()
violin_path = os.path.join(figures_dir, '02_bmi_threat_violin.png')
plt.savefig(violin_path, dpi=300)
plt.close()
print(f"  ✅ Violin plot saved to: {violin_path}")

# (3) Mean Plot with 95% CI
plt.figure(figsize=(8, 6))
sns.pointplot(x='Threat_Group', y='BMIPCT', data=df, errorbar=('ci', 95), capsize=0.1, color='#2C3E50', markers='o', linestyles='-')
plt.title('Mean BMI Percentile with 95% Confidence Interval', fontsize=14, pad=15)
plt.xlabel('Weapon Threat Frequency on School Property', fontsize=12, labelpad=10)
plt.ylabel('Mean BMI Percentile', fontsize=12, labelpad=10)
plt.tight_layout()
meanplot_path = os.path.join(figures_dir, '03_bmi_threat_mean_ci.png')
plt.savefig(meanplot_path, dpi=300)
plt.close()
print(f"  ✅ Mean Plot with 95% CI saved to: {meanplot_path}")


[Step 3] Generating and Saving High-Resolution Visualizations...
  ✅ Boxplot saved to: ../outputs/figures/01_bmi_threat_boxplot.png
  ✅ Violin plot saved to: ../outputs/figures/02_bmi_threat_violin.png
  ✅ Mean Plot with 95% CI saved to: ../outputs/figures/03_bmi_threat_mean_ci.png


In [39]:
# ==========================================================
# 5. Automated Statistical Summary Report & ANOVA Table Export
# ==========================================================
print(f"\n[Step 4] Exporting Final Reports...")

# 1. Export ANOVA test results table to the 'tables' folder
anova_results_df = pd.DataFrame({
    'Test_Name': ['One-Way ANOVA'],
    'F_Statistic': [round(f_stat, 4)],
    'P_Value': [p_val],
    'Significant_At_0.05': [p_val < 0.05]
})
anova_table_path = os.path.join(tables_dir, '02_anova_results.csv')
anova_results_df.to_csv(anova_table_path, index=False)
print(f"  📁 ANOVA results table saved to: {anova_table_path}")

# 2. Generate Markdown summary report to the 'summary' folder
total_n = len(df)
means = desc_stats.set_index('Threat_Group')['Mean_BMI_Pct'].to_dict()
counts = desc_stats.set_index('Threat_Group')['Sample_Size'].to_dict()

report_content = f"""# Final Individual Project: ANOVA Statistical Report

**Dataset:** `cleaned_yrbs_master.csv`
**Research Question:** Weapon Threat Frequency and BMI Percentile (ANOVA)
**Responsible Person:** 李宥宣 (113370237)

## 1. Descriptive Statistics Overview
* **Total Analysis Samples (N):** {total_n} students
* **Group '0 times':** N = {int(counts['0 times'])}, Mean BMI Pct = {means['0 times']:.2f}
* **Group '1 time':** N = {int(counts['1 time'])}, Mean BMI Pct = {means['1 time']:.2f}
* **Group '2+ times':** N = {int(counts['2+ times'])}, Mean BMI Pct = {means['2+ times']:.2f}

## 2. One-Way ANOVA Test Results
* **F-Statistic:** {f_stat:.4f}
* **P-value:** {p_val:.6f}
* **Significance Level (Alpha):** 0.05

## 3. Conclusion
With a P-value of **{p_val:.6f}**, we **reject the null hypothesis (H0)**. 
There is a statistically significant difference in BMI percentiles among US high school students based on how frequently they are threatened or injured with a weapon on school property.
"""

# Define the target path and write to file
report_file_path = os.path.join(summary_dir, 'anova_summary_report.md')
with open(report_file_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"  📄 Statistical summary report successfully created at: {report_file_path}")


[Step 4] Exporting Final Reports...
  📁 ANOVA results table saved to: ../outputs/tables/02_anova_results.csv
  📄 Statistical summary report successfully created at: ../outputs/summary/anova_summary_report.md
